# Load Dependencies


## Files needed to run:

- `feature_columns.pkl`
- `svm_finetuned.pkl`
- `rf_finetuned.pkl`
- `CBERT_checkpoint.pth`
- `EnsembleProcessing.py`
- All files are available in the Final Implementation folder of our [repository](https://github.com/MiguelPartosa/Thesis-FOS-BinaryClass-WSD).


In [14]:
import pandas as pd
import torch
from torch import cuda
from transformers import DistilBertTokenizer, DistilBertModel

## CBERT Checkpoint


In [15]:
device = 'cuda' if cuda.is_available() else 'cpu'


class BertClass(torch.nn.Module):
    def __init__(self):
        super(BertClass, self).__init__()
        self.l1 = DistilBertModel.from_pretrained('GianTan/CBERTo')
        self.pre_classifier = torch.nn.Linear(768, 768)
        self.dropout = torch.nn.Dropout(0.3)

        self.pre_classifier2 = torch.nn.Linear(768, 768)
        self.dropout2 = torch.nn.Dropout(0.3)

        self.classifier = torch.nn.Linear(768, 1)

    def forward(self, input_ids, attention_mask, token_type_ids):
        output_1 = self.l1(input_ids=input_ids, attention_mask=attention_mask)
        hidden_state = output_1[0]
        pooler = hidden_state[:, 0]
        pooler = self.pre_classifier(pooler)
        pooler = torch.nn.Tanh()(pooler)
        pooler = self.dropout(pooler)
        pooler = self.pre_classifier2(pooler)
        pooler = torch.nn.Tanh()(pooler)
        pooler = self.dropout2(pooler)

        output = self.classifier(pooler)
        return output.squeeze(1)


cbert_model = BertClass()
cbert_model.to(device)

tokenizer = DistilBertTokenizer.from_pretrained(
    'GianTan/CBERTo', truncation=True, do_lower_case=False)
optimizer = torch.optim.Adam(params=cbert_model.parameters(), lr=4e-05)

# load
checkpoint = torch.load('CBERT_checkpoint.pth', map_location='cpu')
cbert_model.load_state_dict(checkpoint['model_state_dict'])
optimizer.load_state_dict(checkpoint['optimizer_state_dict'])

## SVM and Random Forest Checkpoint


Importing


In [16]:
import pickle
import pandas as pd
from EnsembleProcessing import process_embeddings
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer

# Dependencies from read pickle file


def GetTextCol(X):
    text_cols = X.select_dtypes(include=['object', 'string']).columns
    if len(text_cols) == 0:
        raise ValueError("No text columns found in input DataFrame")
    # Error is raised if the first index is not returned.
    return text_cols[0]


def GetNumCol(X):
    return X.select_dtypes(include=['int64', 'float64']).columns.tolist()


preprocessor = ColumnTransformer(
    transformers=[
        ("tfidf", TfidfVectorizer(), GetTextCol),
        ("scaler", StandardScaler(), GetNumCol)
    ],
    remainder='drop'  # Remove any unhandled columns
)

# SVM
with open('svm_finetuned.pkl', 'rb') as f:
    svm_best = pickle.load(f)

# Random Forest
with open('rf_finetuned.pkl', 'rb') as f:
    rf_best = pickle.load(f)

c:\Users\User\miniconda3\envs\data_science\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfVectorizer from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\User\miniconda3\envs\data_science\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator StandardScaler from version 1.5.2 when using version 1.6.1. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\User\miniconda3\envs\data_science\Lib\site-packages\sklearn\base.py:380: InconsistentVersionWarning: Trying to unpickle estimator TfidfTransformer from version 1.

# Model Function Backend


## CBERT


In [17]:
def test_model(item):
    input_text = item
    encoded_text = tokenizer.encode_plus(
        input_text,
        None,
        add_special_tokens=True,
        max_length=256,
        pad_to_max_length=True,
        return_token_type_ids=True
    )

    # Convert the input to tensors
    input_ids = torch.tensor(encoded_text['input_ids']).unsqueeze(0)
    input_mask = torch.tensor(encoded_text['attention_mask']).unsqueeze(0)
    segment_ids = torch.tensor(encoded_text['token_type_ids']).unsqueeze(0)

    # Move tensors to the device
    input_ids = input_ids.to(device)
    input_mask = input_mask.to(device)
    segment_ids = segment_ids.to(device)

    # Make predictions
    with torch.no_grad():
        outputs = cbert_model(input_ids, input_mask, segment_ids)

    # Apply sigmoid activation function
    outputs = torch.sigmoid(outputs)

    # Convert the outputs to numpy array
    outputs = outputs.cpu().detach().numpy()

    return outputs

## SVM and Random Forest


In [18]:
# When predicting on new data:
def PredictSample(input_df):
    # Load models and feature columns
    with open('svm_finetuned.pkl', 'rb') as f:
        svm_model = pickle.load(f)

    # Random Forest
    with open('rf_finetuned.pkl', 'rb') as f:
        rf_model = pickle.load(f)

    with open('feature_columns.pkl', 'rb') as f:
        feature_columns = pickle.load(f)

    try:
        # Embeddings
        embeddings_df = process_embeddings(input_df, variance_threshold=1)
        combined_df = pd.concat([embeddings_df, input_df], axis=1)

        # Clean read issue tensor values
        if 'Similarity Scores' in combined_df.columns:
            combined_df['Similarity Scores'] = combined_df['Similarity Scores'].apply(
                lambda x: x.item() if hasattr(x, 'item') else x
            )

        # Drop Non-features
        for col in ['Is FOS', 'Word Sense', 'Verb', 'Usage']:
            if col in combined_df.columns:
                combined_df = combined_df.drop(columns=[col])

        # Align
        # Create a DataFrame with all required columns, filled with zeros
        aligned_row = pd.DataFrame(0, index=[0], columns=feature_columns)

        for col in combined_df.columns:
            if col in feature_columns:
                aligned_row[col] = combined_df[col]

        rf_prob = rf_model.predict_proba(aligned_row)[0][1]
        svm_pred = svm_model.predict(aligned_row)[0]
        return {
            # 'rf_probability': rf_prob,
            'rf_prediction': 1 if rf_prob >= 0.5 else 0,
            'svm_prediction': int(svm_pred)
        }

    except Exception as e:
        # print(f"Error during prediction: {e}")
        print(e)
        # return {'error': str(e)}

In [19]:
import warnings


def prediction_values(score):
    return 'Non-Literal' if score >= 0.5 else 'Literal'

# trained on "is fos" column


def ClassifyExample(df) -> tuple:
    '''
    Returns Cbert, SVM, and RF
    '''
    cbert_result = prediction_values(test_model(df['Usage'][0]))
    with warnings.catch_warnings(action="ignore"):
        models_result = PredictSample(df)
    svm_result = prediction_values(models_result['svm_prediction'])
    rf_result = prediction_values(models_result['rf_prediction'])
    return (cbert_result, svm_result, rf_result)

# Experimentation

## Transforming Dataset

- Transform three input columns into set format for function of predicting usage by merging with their word sense from the original dataset


In [ ]:
example_df = pd.read_csv(
    './Experimentation Implementation/example_filled_table.csv')
example_df

,Verbs,Literal Example,Non-Literal Example
0,bantay,Ang iro nagbantay sa balay sa gabii.,Bantay lang ka kung madakpan ka sa imong gibuhat!
1,mogawas,Mogawas ko sa balay aron mopalit og pan.,Mogawas ra ang tinuod nga intensyon sa taas ng...
2,pukawon,Gipukaw nako siya kay sayo na sa buntag.,Gipukaw sa trahedya ang ilang kaisog ug panagh...
3,ul-ol,Ang bata nag-ul-ol sa iyang dulaan.,"Ayaw pag-ul-ol anang butanga, seryoso ni!"
4,"pagtan-aw, kitkit, paglingi",Naglingi-lingi siya aron pangitaon ang iyang h...,"Ayaw kitkita ang kagahapon, magpadayon ta sa u..."
5,lupad,Ang langgam nilupad paingon sa kahoy.,"Nilupad ang akong damgo, wala matuman."
6,luhod,Nagluhod siya samtang nag-ampo sa simbahan.,Luhod nalang ta ani kung magpadayon ang problema.
7,punit,Gipunit nako ang lapis nga nahulog sa salog.,Nagpunit siya og maayong oportunidad sa iyang ...
8,bitay,Gibitay niya ang iyang sanina sa pisi.,Nabitay sa kawad-on ang ilang panginabuhi.
9,hunong,Mihunong ang sakyanan sa pedestrian lane.,"Hunonga ang imong mga bakak, daghan na kaayo!"


In [ ]:
def TransformDataset(df, user_id:str) -> pd.DataFrame:
    # Prep output dataset
    transformed_columns = ['User ID', 'Verb', 'Sentence Type', 'Sentence']
    transformed_df = pd.DataFrame(columns=transformed_columns)

    # Process dataset into examples. 30 rows result
    for row in example_df.itertuples():
        transformed_df = pd.concat([transformed_df, 
                                    pd.DataFrame({
                                        'User ID': user_id, 'Verb': row[1], 'Sentence Type': 'Literal', 'Sentence': row[2]}, index=[row[0]*2]),
                                    pd.DataFrame(
                                        {'User ID': user_id, 'Verb': row[1], 'Sentence Type': 'Non-Literal', 'Sentence': row[3]}, index=[row[0]*2+1])
                                    ])
    return transformed_df


example_output = TransformDataset(example_df, 'Mama ni Giordan')
example_output

,User ID,Verb,Sentence Type,Sentence
0,Mama ni Giordan,bantay,Literal,Ang iro nagbantay sa balay sa gabii.
1,Mama ni Giordan,bantay,Non-Literal,Bantay lang ka kung madakpan ka sa imong gibuhat!
2,Mama ni Giordan,mogawas,Literal,Mogawas ko sa balay aron mopalit og pan.
3,Mama ni Giordan,mogawas,Non-Literal,Mogawas ra ang tinuod nga intensyon sa taas ng...
4,Mama ni Giordan,pukawon,Literal,Gipukaw nako siya kay sayo na sa buntag.
5,Mama ni Giordan,pukawon,Non-Literal,Gipukaw sa trahedya ang ilang kaisog ug panagh...
6,Mama ni Giordan,ul-ol,Literal,Ang bata nag-ul-ol sa iyang dulaan.
7,Mama ni Giordan,ul-ol,Non-Literal,"Ayaw pag-ul-ol anang butanga, seryoso ni!"
8,Mama ni Giordan,"pagtan-aw, kitkit, paglingi",Literal,Naglingi-lingi siya aron pangitaon ang iyang h...
9,Mama ni Giordan,"pagtan-aw, kitkit, paglingi",Non-Literal,"Ayaw kitkita ang kagahapon, magpadayon ta sa u..."


In [ ]:
predict_output_cols = ['CBERT Prediction',
                       'SVM Prediction', 'Random Forest Prediction']

In [ ]:
# TODO Params
# @title Enter new parameters and rerun cell to refresh results. { display-mode: "form" }
from IPython.display import HTML, display, Javascript

test_fos = 'makabuhi og patay'  # @param {type:"string"}
# @param {type:"string"}
test_word_sense = 'usa ka makapatikod nga bakak o sugilanon'
test_verb = 'Makabuhi, Patay'  # @param {type:"string"}

# @param {type:"string"}
test_usage = 'Ang mga tawo nga makabuhi og patay kasagaran maayo kaayo mamatay.'
# @param ["Non-literal Example", "Literal Example"]
test_is_fos = 'Non-literal Example'

test_is_fos = 0 if test_is_fos == "Literal Example" else 1

test_df = pd.DataFrame({'FOS': [test_fos], 'Word Sense': [test_word_sense], 'Verb': [
                       test_verb], 'Usage': [test_usage], 'Is FOS': [test_is_fos]})

prediction_output = ClassifyExample(test_df)
print('\n\n')
df = pd.DataFrame({
    "Model": ["Cbert", "SVM", "Random Forest"],
    "Prediction": prediction_output
})

html = f"""
            <div style="scale: 100%;">
                {df.to_html(index=False, border=1)}
            </div>
        """

display(HTML(html))
Javascript("google.colab.output.setIframeHeight('500px');")

c:\Users\Miguel\Documents\Project Source Files\IT Work\School\Thesis\.venv\Lib\site-packages\transformers\tokenization_utils_base.py:2834: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(


Generating embeddings...


Transforming Usage Embeddings: 100%|██████████| 3/3 [00:00<00:00, 87.96it/s]


Number of components for 100% variance:
Verb: 768 components
Usage: 768 components
Sentence: 768 components
Optimal number of clusters:
Verb: 2 clusters
Usage: 2 clusters
Sentence: 2 clusters





Model,Prediction
Cbert,Non-Literal
SVM,Non-Literal
Random Forest,Literal


<IPython.core.display.Javascript object>